# Predict (Shannon) Entropy

## Introduction

In the previous section, we showed that mean annual discharge could be predicted well from catchment attributes.  In this step we aim to predict a measure of the randomness of river systems from catchment attributes, also known as entropy (H).  Since the (Shannon) entropy of the distribution does not embody one specific process, it does not fit with conventional classifications of hydrological signatures.  Since the entropy measure encompasses the entire distribution, it can be interpreted as an aggregate representation of the complex interactions of the hydrologic cycle.

In the data preprocessing, we computed the entropy of the distribution of each individual streamflow time series in bits per sample.  We'll now use an ensemble decision tree method called XGBoost (eXtreme Gradient Boosted decision tree) {cite}`chen2016xgboost` to see if the entropy (or uncertainty) of a distribution can be predicted from catchment attributes.  The dictionary size (number of quantization levels) is varied to test if the additional information in the distribution can be exploited by the model.  The model input features are added in successive model tests to compare the contribution of catchment attribute groups related to climate, terrain, land cover, and soil.  

In [2]:
import os, sys
import pandas as pd
import numpy as np
import geopandas as gpd

import xgboost as xgb
from sklearn.metrics import (
    r2_score
)
from pathlib import Path    
from scipy.stats import linregress

# Add repo root to path
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import Config

from utils import data_processing_functions as dpf

from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, Whisker, HoverTool
from bokeh.io import output_notebook
from bokeh.palettes import Sunset10, Vibrant7
from pathlib import Path

output_notebook()

BASE_DIR = Path(os.getcwd())
BASE_DIR

Loading BokehJS ...

PosixPath('/home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks')

## Load Input Data

In [3]:
# load catchment attributes
attributes_filename = 'Watershed_descriptors_20260203_with_stats.csv'
# attributes_filename = 'BCUB_watershed_attributes_updated_20250227_centroids.geojson'
gdf = gpd.read_file(BASE_DIR / 'data' / attributes_filename)
gdf.columns = [c.lower() for c in gdf.columns]
# drop the Unnamed: 0 column
gdf.drop(columns=['unnamed: 0'], inplace=True, errors='ignore')
gdf.rename(columns=Config.CLIMATE_COLUMN_MAPPER, inplace=True)

# stations excluded for quality issues
exclude_stations = Config.EXCLUDED_STATIONS

df = gdf[[c for c in gdf.columns if c not in ['geometry']]]
df = df[~df['official_id'].isin(exclude_stations)]
df['official_id'] = df['official_id'].astype(str)
df.set_index('official_id', inplace=True)
len(df)

1300

In [4]:
# filter for the catchment set with FDC prediction results processed
fdc_estimation_results_folder = Path('/home/danbot/code/distribution_estimation/docs/notebooks/data/results/fdc_estimation_results/kde_results')
knn_results_folder = fdc_estimation_results_folder / 'knn'
result_files = os.listdir(knn_results_folder)
result_files = [f for f in result_files if f.endswith('.json')]
processed_catchments = [f.split('_')[0] for f in result_files]
print(len(processed_catchments), 'catchments with processed FDC estimation results')

# filter to only predict the processed catchments
df = df[df.index.isin(processed_catchments)]


706 catchments with processed FDC estimation results


In [5]:
def compute_entropy(p):
    mask = p > 0  # Remove zero probabilities
    return -np.sum(p[mask] * np.log2(p[mask]))

In [6]:
# load the target data (entropy computed for each catchment POR pmf)
bits = [6, 8, 10]
entropy_output_folder = BASE_DIR /  'data' / 'pmf_entropy'
os.makedirs(entropy_output_folder, exist_ok=True)
entropy_dict = {}
for b in bits:
    entropy_fpath = entropy_output_folder / f'pmf_entropy_{b:02d}_bits.csv'
    if not entropy_fpath.exists():
        entropy_df = pd.DataFrame()
        baseline_pmf_folder = BASE_DIR / 'data' / 'baseline_distributions' / f'{b:02d}_bits'
        f = f'pmf_kde_adaptive_mixture.csv'
        sample_pmfs = pd.read_csv(baseline_pmf_folder / f, index_col=['log_x_uar'])
        # assert all pmf columns sum to 1
        column_sums = sample_pmfs[processed_catchments].sum(axis=0).values
        assert np.allclose(column_sums, 1), np.max(np.abs(1 - column_sums))
        # compute column-wise entropy
        entropy_df[f'{b:02d}_bits'] = sample_pmfs[processed_catchments].apply(compute_entropy, axis=0)
        # assert that all columns have valid entropy values
        assert entropy_df[f'{b:02d}_bits'].notna().all(), f"Invalid entropy values found in {b:02d}_bits"
        entropy_df['official_id'] = entropy_df.index.astype(str)
        entropy_df.to_csv(entropy_output_folder / f'pmf_entropy_{b:02d}_bits.csv', index=False)
    else:
        entropy_df = pd.read_csv(entropy_fpath, dtype={'official_id': str})
        
    entropy_dict[f'{b:02d}_bits'] = entropy_df


In [7]:
from utils.plotting import apply_tufte_style
# plot the distribution of entropy values across catchments for each bit level
p_list = []
for b in bits:
    entropy_df = entropy_dict[f'{b:02d}_bits']
    p = figure(title=f'Entropy Distribution for {b:02d}-bit PMFs', x_axis_label='Entropy (bits)', y_axis_label='Density')
    hist, edges = np.histogram(entropy_df[f'{b:02d}_bits'], density=True, bins=40)
    p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], fill_color=Sunset10[0], line_color='black', alpha=0.7)
    p_list.append(p)
    apply_tufte_style(p)
show(gridplot(p_list, ncols=3, width=400, height=300))

/home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks/utils/plotting.py:124: UserWarning: 
You are attempting to set `plot.legend.label_text_font` on a plot that has zero legends added, this will have no effect.

Before legend properties can be set, you must add a Legend explicitly, or call a glyph method with a legend parameter set.

  fig.legend.label_text_font = font
/home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks/utils/plotting.py:125: UserWarning: 
You are attempting to set `plot.legend.label_text_font_size` on a plot that has zero legends added, this will have no effect.

Before legend properties can be set, you must add a Legend explicitly, or call a glyph method with a legend parameter set.

  fig.legend.label_text_font_size = legend_text_font_size


Subdivide the attributes into related classes: terrain, land cover, soil, climate.

## Define Attribute Groups

In [8]:
terrain = ['drainage_area_km2', 'elevation_m', 'slope_deg', 'aspect_deg']
land_cover = [
    'land_use_forest_frac_2010', 'land_use_grass_frac_2010', 'land_use_wetland_frac_2010', 'land_use_water_frac_2010', 
    'land_use_urban_frac_2010', 'land_use_shrubs_frac_2010', 'land_use_crops_frac_2010', 'land_use_snow_ice_frac_2010']
soil = ['logk_ice_x100', 'porosity_x100']
raw_climate = ['prcp (mm/day)', 'high_prcp_freq (fraction)',
       'low_prcp_freq (fraction)', 'high_prcp_duration (days)',
       'low_prcp_duration (days)', 'tmax (degrees c)', 'tmin (degrees c)',
       'vp (pa)', 'pet (mm/day)', 'swe (kg/m2)', 'srad (w/m2)', 'dayl (s)']
climate = [Config.CLIMATE_COLUMN_MAPPER[c] for c in raw_climate]
all_attributes = terrain + land_cover + soil + climate
assert len([c for c in all_attributes if c not in df.columns]) == 0

attribute_set_dict = {
    'climate': climate, 
    '+land_cover': land_cover,
    '+terrain': terrain, 
    '+soil': soil,
}
results_folder = os.path.join(BASE_DIR, 'results', 'entropy_prediction_results')
print(results_folder)
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

/home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks/results/entropy_prediction_results


## Set Trial Parameters

In [9]:
# define the amount of data to set aside for final testing
# holdout_pct = 0.10
nfolds = 5
n_boost_rounds = {4: 1500, 6: 1000, 8: 1500, 10: 1500,}
n_optimization_rounds = 10
loss = 'reg:absoluteerror'
# loss = 'reg:squarederror'  # use squared error for regression
# min_years_record = 5

# attribute_set_names = ['climate', '+land_cover', '+terrain', '+soil']
attribute_set_names = ['+soil',  '+land_cover', '+terrain', 'climate']
# active_attribute_set_names = ['climate', '+land_cover', '+terrain', '+soil']  # sample run to avoid the full cumulative sweep
eval_metrics = {'reg:squarederror': 'test_rmse', 'reg:absoluteerror': 'test_mae'}

In [11]:
from utils.xgb_functions import run_xgb_CV_trials

results_dict = {}
group_results_dict = {}
for bitrate in [6, 8, 10]:
    # set the target column
    # target_column = f'h_{bitrate}_bits'
    target_column = f'{bitrate:02d}_bits'
    eval_metric = eval_metrics[loss]
    input_attributes = []    
    results_dict[target_column] = {}
    group_results_dict[target_column] = {}
    # add attribute groups successively
    entropy_df = entropy_dict[f'{bitrate:02d}_bits'].set_index('official_id')

    # map all the attributes to the same official_id index
    mapped_entropy_df = pd.concat([entropy_df, df[all_attributes]], axis=1, join='inner').reset_index()

    for n, group in enumerate(attribute_set_names):
        print(f'  [{bitrate} bits] Processing {group} attribute set using {loss} objective.')
        group_attributes = attribute_set_dict[group]
        input_attributes += group_attributes
        input_data =  mapped_entropy_df[['official_id'] + input_attributes + [target_column]].copy()

        group_results_fname = f'{target_column}_entropy_prediction_results_{"".join(group)}.npy'
        group_results_fpath = os.path.join(results_folder, group_results_fname)

        if os.path.exists(group_results_fpath):
            print(f'Retrieving existing results from {group_results_fpath}')
            group_results = np.load(group_results_fpath, allow_pickle=True).item()
        else:
            # normalize the entropy by the bitrate to support
            # comparison across bitrates
            input_data[target_column] /= bitrate * 1.0
            result_df, all_predictions_df, all_convergence_df = run_xgb_CV_trials(
                group, input_attributes, target_column, 
                input_data, n_optimization_rounds, 
                nfolds, n_boost_rounds[bitrate], results_folder, loss, 
            )
            group_results = {
                'all_results': result_df,
                'convergence': all_convergence_df,
                'oos_predictions': all_predictions_df,
            } 
            np.save(group_results_fpath, group_results)

        group_results_dict[target_column][n] = {'order': group, 'results': group_results}
        if group_results['all_results'].empty:
            continue
        
        # Save all results, not just the best
        results_dict[target_column][f'group_{n}'] = {
            'group_order': group,
            'test_error': group_results['all_results'][f'{eval_metric}_mean'],
            'test_error_std': group_results['all_results'][f'{eval_metric}_stdev'],
            'convergence': group_results['convergence'],
            'oos_predictions': group_results['oos_predictions'],
        }
        summary_csv = f'{target_column}_summary_group_{n}.csv'
        group_results['oos_predictions'].to_csv(os.path.join(results_folder, summary_csv), index=False)


  [6 bits] Processing +soil attribute set using reg:absoluteerror objective.
Retrieving existing results from /home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks/results/entropy_prediction_results/06_bits_entropy_prediction_results_+soil.npy
  [6 bits] Processing +land_cover attribute set using reg:absoluteerror objective.
Retrieving existing results from /home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks/results/entropy_prediction_results/06_bits_entropy_prediction_results_+land_cover.npy
  [6 bits] Processing +terrain attribute set using reg:absoluteerror objective.
Retrieving existing results from /home/danbot/code/2025/divergence_prediction/divergence_prediction/notebooks/results/entropy_prediction_results/06_bits_entropy_prediction_results_+terrain.npy
  [6 bits] Processing climate attribute set using reg:absoluteerror objective.
Retrieving existing results from /home/danbot/code/2025/divergence_prediction/divergence_prediction/no

## View Results

In [12]:
def create_results_plots(target_col, data, eval_metric, title=''):    
    plots = []
    metric_label = eval_metric.replace('test_', '').upper()

    def _plot_attribute_order_line(data):
        means, medians, lbs, ubs = [], [], [], []

        group_order = []
        for k, d in data.items(): # k is group index, d is results
            label = d['order']
            print(d['results'].keys())
            df = d['results']['all_results']
            # df = d['results']['oos_predictions']
            trial_means = df[f'{eval_metric}_mean'].values
            # trial_medians = df[f'{eval_metric}_median'].values
            means.append(np.mean(trial_means))
            # medians.append(np.median(trial_medians))
            lb, ub = np.percentile(trial_means, [2.5, 97.5])
            lbs.append(lb)
            ubs.append(ub)
            group_order.append(label)


        max_range = (0.95*min(lbs), max(ubs)*1.05)
        source = ColumnDataSource({'x': group_order, 'y1': means, 'ub': ubs, 'lb': lbs})    
        fig = figure(title='', x_range=group_order, y_range=max_range)
        fig.scatter('x', 'y1', legend_label=metric_label, color='green', source=source, line_width=3, marker='square', size=4)
        fig.add_layout(Whisker(source=source, base='x', upper='ub', lower='lb', line_width=1))
        fig.legend.background_fill_alpha = 0.6
        fig.yaxis.axis_label = rf'$$\text{{{metric_label}}}$$'
        fig.xaxis.axis_label = r'$$\text{Attribute Groups}$$'
        # best_set = min(attribute_sets, key=lambda x: results_df[x]['all_results'][f'{eval_metric}_mean'].mean())
        return fig#, best_set

    def _plot_scatter_with_regression(best_result_df, xlabel, ylabel, target_col):
        trial_r2 = (
            best_result_df.groupby('trial')[['actual', 'predicted']]
            .apply(lambda df: r2_score(df['actual'], df['predicted']))
        )
        # r2_mean = trial_r2.mean()
        # get the median performing trial?  no -- this is data leakage
        r2_std = trial_r2.std()
        grouped = best_result_df.groupby('trial').agg({
            'actual': 'median',
            'predicted': 'median',
        })
        grouped['diff'] = (grouped['actual'] - grouped['predicted']).abs()
        median_trial = grouped.sort_values('diff').index[len(grouped) // 2]
        best_result = best_result_df[best_result_df['trial'] == median_trial].copy()
        xx, yy = best_result['actual'].values, best_result['predicted'].values
        source = ColumnDataSource({'x': xx, 'y': yy, 'ID': best_result['official_id'].values})
        slope, intercept, r, p, se = linregress(xx, yy)

        sfig = figure(title='')
        sfig.scatter('x', 'y', size=3, alpha=0.8, source=source, legend_label=target_col, color='blue')
        sfig.add_tools(HoverTool(tooltips=[('ID', '@ID')]))
        x_obs = np.linspace(min(xx), max(xx), 1000)
        ybf = [slope * e + intercept for e in x_obs]
        sfig.line(x_obs, ybf, color='red', line_width=3, line_dash='dashed', legend_label=f'R²={r**2:.2f} ± {r2_std:.3f}')
        sfig.line([min(x_obs), max(ybf)], [min(x_obs), max(ybf)], color='black', line_dash='dotted', line_width=2, legend_label='1:1')
        sfig.xaxis.axis_label = xlabel
        sfig.yaxis.axis_label = ylabel
        sfig.legend.background_fill_alpha = 0.5
        sfig.legend.location = 'bottom_right'
        return sfig, median_trial

    def _plot_convergence(convergence_df, median_trial):
        cfig = figure(title='')

        data = convergence_df[convergence_df['trial'] == median_trial].copy()
        fold_nos = sorted(set(convergence_df['fold']))

        min_pred_risk = 1e9
        for fn in fold_nos:
            fold_data = data[data['fold'] == fn].copy()
            cfig.line(fold_data['round'], fold_data[f'test'], line_alpha=0.6, line_color='red', line_dash='dotted', legend_label=f'Test Folds')
            cfig.line(fold_data['round'], fold_data[f'train'], line_alpha=0.7, line_color='grey', line_dash='dotted', legend_label=f'Train Folds')

        train_pivot = data.pivot_table(index='round', columns='fold', values='train')#, aggfunc='first')
        test_pivot = data.pivot_table(index='round', columns='fold', values='test')#, aggfunc='first')
        train_pivot.columns = [f'fold_{col}' for col in train_pivot.columns]
        test_pivot.columns = [f'fold_{col}' for col in test_pivot.columns]

        train_pivot['mean'] = train_pivot.mean(axis=1)
        test_pivot['mean'] = test_pivot.mean(axis=1)
        cfig.line(train_pivot.index, train_pivot['mean'], line_alpha=0.5, line_color='grey', line_width=2, legend_label='CV Mean (Train)')
        cfig.line(test_pivot.index, test_pivot['mean'], line_alpha=0.5, line_color='red', line_width=2, legend_label='CV Mean (Test)')

        min_pred_risk_idx = test_pivot['mean'].idxmin()
        min_pred_risk = test_pivot.loc[min_pred_risk_idx, 'mean']
        if min_pred_risk_idx == max(test_pivot['mean'].index):
            print(f'Min prediction risk occurs at the maximum iteration, try increasing the number of boosting rounds')
        cfig.line([min_pred_risk_idx, min_pred_risk_idx], [0, min_pred_risk], legend_label='Min risk', color='green', line_width=2, line_dash='dashed')
        cfig.legend.location = 'top_right'
        cfig.legend.background_fill_alpha = 0.5
        cfig.xaxis.axis_label = r'$$\text{Iteration}$$'
        cfig.yaxis.axis_label = rf'$$\text{{{metric_label}}}$$'
        return cfig

    def _plot_target_cdfs(cdf_arrays, xlabel):
        cdffig = figure(title='', x_axis_type='log')
        for (cdfx, cdfy) in cdf_arrays:
            cdffig.line(cdfx, cdfy, color='black', line_alpha=0.6, line_width=2, legend_label='Fold CDFs')
        cdffig.xaxis.axis_label = xlabel
        cdffig.yaxis.axis_label = r'$$\text{Pr}(X\leq x) $$'
        cdffig.legend.location = 'top_left'
        return cdffig
    
    def _compute_empirical_cdf(data):
        sorted_data = np.sort(data)
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        return sorted_data, cdf

    # # Plot 1: RMSE/MAE across attribute sets
    attr_fig = _plot_attribute_order_line(data)
    # attr_fig = _plot_attribute_order_cdf()
    plots.append(attr_fig)

    # Plot 2: Actual vs Predicted for best set
    best_attr = list(data.keys())[0]
    print(data.keys())
    best_result = data[best_attr]['results']['oos_predictions']
    sfig, median_trial = _plot_scatter_with_regression(best_result, r'$$H [\text{bits}]$$', r'$$\hat H [\text{bits}]$$', target_col)
    plots.append(sfig)

    # # Plot 3: Convergence plot
    convergence_df = data[best_attr]['results']['convergence']
    cfig = _plot_convergence(convergence_df, median_trial)
    plots.append(cfig)

    # Plot 4: CDFs
    cdf_arrays = []
    for i, grp_result in data[best_attr]['results']['oos_predictions'].groupby('fold'):
        sorted_data, cdf = _compute_empirical_cdf(grp_result['actual'].values)
        cdf_arrays.append((sorted_data, cdf))
    cdffig = _plot_target_cdfs(cdf_arrays, r'$$H [\text{bits}]$$')
    plots.append(cdffig)

    return plots

In [13]:
n = 0
all_plots = []
eval_metrics = {'reg:squarederror': 'test_rmse', 'reg:absoluteerror': 'test_mae'}

n = 0
all_plots = []
for c in [f'{b:02d}_bits' for b in [6, 8, 10]]:
    data = group_results_dict[c]
    eval_metric = eval_metrics[loss]
    first_result = next(iter(data.values()))
    grp_plots = create_results_plots(c, data, eval_metric)
    all_plots += grp_plots

layout = gridplot(all_plots, ncols=4, width=300, height=275)
show(layout)

dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys([0, 1, 2, 3])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys([0, 1, 2, 3])
Min prediction risk occurs at the maximum iteration, try increasing the number of boosting rounds
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys(['all_results', 'convergence', 'oos_predictions'])
dict_keys([0, 1, 2, 3])


## Discussion

Model performance metrics are compared across different dictionary sizes by normalizing by the max entropy ($\log_2(2^b)$).

An important question about the model results, or the unexplained variance in the predictability of entropy, is whether the residuals correlate with the predictability of other, independent signatures.  Below we look at two things -- the first is the correlation of mean runoff and entropy, and the correlation of residuals between the two models.  Is it the same catchments that are difficult to predict, regardless of the target variable, and likewise is it the same catchments that are predicted well?

The lack of correlation between mean runoff and entropy prediction residuals suggests the two target variables are independent.  Mean runoff captures a central tendency, while entropy captures the system variability.  These variables are independent, or the model could simply be limited in its capacity to capture the interdependence between the variability of the system and its average behaviour.

## Citations

```{bibliography}
:filter: docname in docnames
```